In [ ]:
import os
import asyncio
import gspread
from dotenv import load_dotenv
from oauth2client.service_account import ServiceAccountCredentials
from aiogram import Bot, Dispatcher, types
from aiogram.filters import Command
from aiogram.fsm.context import FSMContext
from aiogram.fsm.state import State, StatesGroup
from aiogram.types import InlineKeyboardMarkup, InlineKeyboardButton
from google import genai
from google.genai import types as genai_types
from pydantic import BaseModel, Field



In [ ]:

# טעינת משתני סביבה מקובץ .env
load_dotenv()

TELEGRAM_TOKEN = os.getenv("TELEGRAM_TOKEN")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_SHEET_NAME = "שם קובץ הגוגל שיטס שלך"
CREDENTIALS_FILE = "credentials.json"

# אתחול הבוט והקליינט של גוגל
bot = Bot(token=TELEGRAM_TOKEN)
dp = Dispatcher()
ai_client = genai.Client(api_key=GEMINI_API_KEY)

# הגדרת המבנה המדויק שאנחנו רוצים לקבל מגוגל (חילוץ ישויות)
class InventoryItem(BaseModel):
    item: str = Field(description="שם הפריט או החלק החסר")
    department: str = Field(description="המחלקה אליה הפריט שייך במפעל/אתר")
    size: str = Field(description="הגודל, המידות או הקוטר של הפריט")
    quantity: str = Field(description="הכמות המבוקשת (כולל יחידות מידה אם צוינו, למשל: 5 שקים, 12 יחידות)")

class OrderFlow(StatesGroup):
    waiting_for_confirmation = State()

# פונקציית כתיבה לגוגל שיטס
def append_to_google_sheet(item, department, size, quantity):
    try:
        scope = ["https://google.com", "https://googleapis.com"]
        creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_FILE, scope)
        client = gspread.authorize(creds)
        sheet = client.open(GOOGLE_SHEET_NAME).sheet1
        sheet.append_row([item, department, size, quantity])
        return True
    except Exception as e:
        print(f"Error writing to Google Sheets: {e}")
        return False

@dp.message(Command("start"))
async def cmd_start(message: types.Message):
    await message.answer(
        "שלום! כדי לדווח על חוסר או פריטים להזמנה, פשוט כתוב לי בטקסט חופשי.\n\n"
        "💡 *לדוגמה:* 'צריך למחלקת חשמל 40 נורות לד בגודל סטנדרטי'"
    )

@dp.message()
async def handle_order_details(message: types.Message, state: FSMContext):
    user_text = message.text
    await message.answer("💡 מנתח את ההודעה באמצעות Gemini...")

    prompt = f"חלץ את פרטי המלאי מתוך הודעת העובד הבאה: {user_text}"

    try:
        # פנייה ל-Gemini עם דרישה לפלט מובנה (Structured Output)
        response = ai_client.models.generate_content(
            model='gemini-1.5-flash', # מודל ה-Free Tier המהיר
            contents=prompt,
            config=genai_types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=InventoryItem, # המודל מחויב לענות בדיוק במבנה ה-Pydantic שהגדרנו
            ),
        )
        
        # המרת הפלט החזר מ-Gemini חזרה לדיקשנרי של פייתון
        import json
        extracted_data = json.loads(response.text)
        await state.update_data(extracted_data=extracted_data)

        # יצירת כפתורי אישור וביטול
        keyboard = InlineKeyboardMarkup(inline_keyboard=[
            [
                InlineKeyboardButton(text="👍 אשר ושמור באקסל", callback_data="confirm_save"),
                InlineKeyboardButton(text="❌ ביטול", callback_data="cancel_save")
            ]
        ])

        summary_text = (
            f"📋 *אנא ודא את פרטי הדיווח (מנוע Gemini):*\n\n"
            f"📦 *פריט:* {extracted_data.get('item', 'לא צוין')}\n"
            f"🏢 *מחלקה:* {extracted_data.get('department', 'לא צוין')}\n"
            f"📏 *גודל:* {extracted_data.get('size', 'לא צוין')}\n"
            f"🔢 *כמות:* {extracted_data.get('quantity', 'לא צוין')}\n\n"
            f"האם הנתונים נכונים ומאושרים לכתיבה לאקסל?"
        )
        
        await message.answer(summary_text, parse_mode="Markdown", reply_markup=keyboard)
        await state.set_state(OrderFlow.waiting_for_confirmation)

    except Exception as e:
        print(f"Gemini Error: {e}")
        await message.answer("מצטער, חלה שגיאה בניתוח ההודעה. נסה לנסח שוב בצורה ברורה יותר.")

@dp.callback_query(OrderFlow.waiting_for_confirmation)
async def process_confirmation(callback_query: types.CallbackQuery, state: FSMContext):
    if callback_query.data == "confirm_save":
        user_data = await state.get_data()
        data = user_data['extracted_data']
        
        success = append_to_google_sheet(
            item=data.get('item', 'לא צוין'),
            department=data.get('department', 'לא צוין'),
            size=data.get('size', 'לא צוין'),
            quantity=data.get('quantity', 'לא צוין')
        )

        if success:
            await callback_query.message.edit_text("✅ הנתונים נרשמו בהצלחה בשורה חדשה ב-Google Sheets (ללא עלות API)!")
        else:
            await callback_query.message.edit_text("❌ חלה שגיאה טכנית בכתיבה לגיליון. אנא ודא שקובץ ה-credentials תקין.")
    
    elif callback_query.data == "cancel_save":
        await callback_query.message.edit_text("❌ הפעולה בוטלה. הנתונים לא נשמרו.")
    
    await state.clear()

if __name__ == "__main__":
    asyncio.run(dp.start_polling(bot))